# 🐾 EDA v2 — Velocidad de Adopción de Mascotas en Malasia
### Dataset: PetFinder Malaysia · Variable objetivo: `AdoptionSpeed`

| Campo | Detalle |
|---|---|
| 📋 Dataset | `train.csv` — PetFinder Malaysia (Kaggle) |
| 📐 Dimensiones | 14.993 filas × 24 columnas |
| 🎯 Variable objetivo | `AdoptionSpeed` (0 = mismo día → 4 = no adoptado en 100 días) |
| 🛠️ Herramientas | Python · Pandas · Plotly |
| 🗺️ País | Malasia (14 estados) |

---

### 🎯 Escala de AdoptionSpeed

| Valor | Significado | Registros |
|---|---|---|
| **0** | Adoptado el mismo día | 410 (2.7%) |
| **1** | Adoptado en 1–7 días | 3.090 (20.6%) |
| **2** | Adoptado en 8–30 días | 4.037 (26.9%) |
| **3** | Adoptado en 31–90 días | 3.259 (21.7%) |
| **4** | No adoptado en 100+ días | 4.197 (28.0%) |

---
## 📑 Contenido

| # | Sección |
|---|---|
| 1 | ⚙️ Instalación y configuración |
| 2 | 📂 Carga, merge con labels y limpieza |
| 3 | 📋 Estadísticas descriptivas |
| 4 | 🔍 Calidad de datos y nulos |
| 5 | 📊 Frecuencias con histogramas |
| 6 | 🎯 Análisis de AdoptionSpeed |
| 7 | 🎻 Violins: distribuciones por variable categórica |
| 8 | 🗺️ Heatmaps de correlación (simple + parcial) |
| 9 | 🔵 Pairplot interactivo (8 variables) |
| 10 | 🗺️ Análisis geográfico por estado |
| 11 | 📌 Conclusiones |


## ⚙️ 1 · Instalación y Configuración

In [12]:
# Ejecutar solo si falta alguna librería
# import sys
# !{sys.executable} -m pip install pandas plotly


In [13]:
import os, warnings, itertools
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.max_columns", 30)

# ── Paleta principal ─────────────────────────────────────────────
SPEED_COLORS = {
    0: "#2ecc71",   # verde  — mismo día
    1: "#27ae60",   # verde oscuro — 1-7d
    2: "#f39c12",   # naranja — 8-30d
    3: "#e67e22",   # naranja oscuro — 31-90d
    4: "#e74c3c",   # rojo — +100d
}
SPEED_LABELS = {
    0: "0 — Mismo día",
    1: "1 — 1 a 7 días",
    2: "2 — 8 a 30 días",
    3: "3 — 31 a 90 días",
    4: "4 — +100 días",
}
COLOR_DOG  = "#4E79A7"
COLOR_CAT  = "#F28E2B"
BG_COLOR   = "#F8F9FA"
PALETTE    = px.colors.qualitative.Bold

print("✅ Librerías y configuración cargadas")


✅ Librerías y configuración cargadas


In [14]:
import os


BASE = os.getcwd()


print(f"📂 Directorio de trabajo: {BASE}")

# ── Paths relativos a BASE ───────────────────────────────────────
TRAIN_PATH = os.path.join(BASE, "train", "train.csv")
BREED_PATH = os.path.join(BASE, "breed_labels.csv")
COLOR_PATH = os.path.join(BASE, "color_labels.csv")
STATE_PATH = os.path.join(BASE, "state_labels.csv")

# ── Verificación automática ──────────────────────────────────────
print()
all_ok = True
for name, path in [("train.csv",        TRAIN_PATH),
                   ("breed_labels.csv",  BREED_PATH),
                   ("color_labels.csv",  COLOR_PATH),
                   ("state_labels.csv",  STATE_PATH)]:
    exists = os.path.exists(path)
    status = "✅" if exists else "❌  NO ENCONTRADO"
    print(f"  {status}  {name}")
    if not exists:
        all_ok = False

print()
if all_ok:
    print("✅ Todos los archivos encontrados — listo para continuar")
else:
    print("⚠️  Archivos faltantes. Opciones:")
    print(f"   1. Copiá los CSVs a: {BASE}")
    print(f"   2. Cambiá BASE a la carpeta donde están los archivos")


📂 Directorio de trabajo: c:\Users\juanc\OneDrive\Desktop\Pet Finder

  ✅  train.csv
  ✅  breed_labels.csv
  ✅  color_labels.csv
  ✅  state_labels.csv

✅ Todos los archivos encontrados — listo para continuar


## 📂 2 · Carga, Merge con Labels y Limpieza

Integramos `train.csv` con los archivos de razas, colores y estados para obtener **nombres reales** en todos los gráficos.

In [15]:
# Carga de archivos
df     = pd.read_csv(TRAIN_PATH)
breeds = pd.read_csv(BREED_PATH)
colors = pd.read_csv(COLOR_PATH)
states = pd.read_csv(STATE_PATH)

print(f"train  : {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"breeds : {breeds.shape[0]} razas | colors: {colors.shape[0]} colores | states: {states.shape[0]} estados")
df.head(3)


train  : 14,993 filas × 24 columnas
breeds : 307 razas | colors: 7 colores | states: 15 estados


,Type,Name,Age,Breed1,Breed2,Gender,Color1,Color2,Color3,MaturitySize,FurLength,Vaccinated,Dewormed,Sterilized,Health,Quantity,Fee,State,RescuerID,VideoAmt,Description,PetID,PhotoAmt,AdoptionSpeed
0,2,Nibble,3,299,0,1,1,7,0,1,1,2,2,2,1,1,100,41326,8480853f516546f6cf33aa88cd76c379,0,Nibble is a 3+ month old ball of cuteness. He ...,86e1089a3,1.00,2
1,2,No Name Yet,1,265,0,1,1,2,0,2,2,3,3,3,1,1,0,41401,3082c7125d8fb66f7dd4bff4192c8b14,0,I just found it alone yesterday near my apartm...,6296e909a,2.00,0
2,1,Brisco,1,307,0,1,2,7,0,2,2,1,1,2,1,1,0,41326,fa90fa5b1ee11c86938398b60abc32cb,0,Their pregnant mother was dumped by her irresp...,3422e4906,7.00,3


In [16]:
# ── Merge con labels (nombres reales) ───────────────────────────
df = df.merge(
    breeds[['BreedID','BreedName']].rename(columns={'BreedID':'Breed1','BreedName':'BreedName1'}),
    on='Breed1', how='left'
)
df = df.merge(
    colors[['ColorID','ColorName']].rename(columns={'ColorID':'Color1','ColorName':'ColorName1'}),
    on='Color1', how='left'
)
df = df.merge(
    states[['StateID','StateName']].rename(columns={'StateID':'State'}),
    on='State', how='left'
)

# ── Decodificación de variables categóricas ──────────────────────
TYPE_MAP       = {1:"Perro 🐕", 2:"Gato 🐈"}
GENDER_MAP     = {1:"Macho", 2:"Hembra", 3:"Mixto"}
VACCINATED_MAP = {1:"Sí", 2:"No", 3:"No seguro"}
HEALTH_MAP     = {1:"Saludable", 2:"Lesión leve", 3:"Lesión grave"}
MATURITY_MAP   = {1:"Pequeño", 2:"Mediano", 3:"Grande", 4:"Extra grande"}
FUR_MAP        = {1:"Corto", 2:"Mediano", 3:"Largo"}

df["Type_Label"]       = df["Type"].map(TYPE_MAP)
df["Gender_Label"]     = df["Gender"].map(GENDER_MAP)
df["Vaccinated_Label"] = df["Vaccinated"].map(VACCINATED_MAP)
df["Dewormed_Label"]   = df["Dewormed"].map(VACCINATED_MAP)
df["Sterilized_Label"] = df["Sterilized"].map(VACCINATED_MAP)
df["Health_Label"]     = df["Health"].map(HEALTH_MAP)
df["Maturity_Label"]   = df["MaturitySize"].map(MATURITY_MAP)
df["Fur_Label"]        = df["FurLength"].map(FUR_MAP)
df["Speed_Label"]      = df["AdoptionSpeed"].map(SPEED_LABELS)
df["Speed_Color"]      = df["AdoptionSpeed"].map(SPEED_COLORS)
df["HasName"]          = df["Name"].notna().map({True:"Con nombre", False:"Sin nombre"})
df["HasVideo"]         = (df["VideoAmt"] > 0).map({True:"Con video", False:"Sin video"})
df["FeeCategory"]      = pd.cut(df["Fee"],
                                bins=[-1, 0, 50, 200, 500, 3000],
                                labels=["Gratis","Bajo (1-50)",
                                        "Medio (51-200)","Alto (201-500)","Premium (>500)"])

print(f"✅ Dataset enriquecido: {df.shape[0]:,} filas × {df.shape[1]} columnas")
df[["Type_Label","BreedName1","ColorName1","StateName","AdoptionSpeed","Speed_Label"]].head(5)


✅ Dataset enriquecido: 14,993 filas × 40 columnas


,Type_Label,BreedName1,ColorName1,StateName,AdoptionSpeed,Speed_Label
0,Gato 🐈,Tabby,Black,Selangor,2,2 — 8 a 30 días
1,Gato 🐈,Domestic Medium Hair,Black,Kuala Lumpur,0,0 — Mismo día
2,Perro 🐕,Mixed Breed,Brown,Selangor,3,3 — 31 a 90 días
3,Perro 🐕,Mixed Breed,Black,Kuala Lumpur,2,2 — 8 a 30 días
4,Perro 🐕,Mixed Breed,Black,Selangor,2,2 — 8 a 30 días


## 📋 3 · Estadísticas Descriptivas

In [17]:
NUM_COLS = ["Age","Fee","Quantity","PhotoAmt","VideoAmt",
            "Vaccinated","Dewormed","Sterilized","Health",
            "MaturitySize","FurLength","AdoptionSpeed"]
df[NUM_COLS].describe().round(2)


,Age,Fee,Quantity,PhotoAmt,VideoAmt,Vaccinated,Dewormed,Sterilized,Health,MaturitySize,FurLength,AdoptionSpeed
count,14993.00,14993.00,14993.00,14993.00,14993.00,14993.00,14993.00,14993.00,14993.00,14993.00,14993.00,14993.00
mean,10.45,21.26,1.58,3.89,0.06,1.73,1.56,1.91,1.04,1.86,1.47,2.52
std,18.16,78.41,1.47,3.49,0.35,0.67,0.70,0.57,0.20,0.55,0.60,1.18
min,0.00,0.00,1.00,0.00,0.00,1.00,1.00,1.00,1.00,1.00,1.00,0.00
25%,2.00,0.00,1.00,2.00,0.00,1.00,1.00,2.00,1.00,2.00,1.00,2.00
50%,3.00,0.00,1.00,3.00,0.00,2.00,1.00,2.00,1.00,2.00,1.00,2.00
75%,12.00,0.00,1.00,5.00,0.00,2.00,2.00,2.00,1.00,2.00,2.00,4.00
max,255.00,3000.00,20.00,30.00,8.00,3.00,3.00,3.00,3.00,4.00,3.00,4.00


In [18]:
# KPIs principales
total      = len(df)
n_dogs     = (df["Type"]==1).sum()
n_cats     = (df["Type"]==2).sum()
avg_speed  = df["AdoptionSpeed"].mean()
pct_fast   = (df["AdoptionSpeed"] <= 1).mean() * 100
pct_slow   = (df["AdoptionSpeed"] == 4).mean() * 100
n_states   = df["StateName"].nunique()
avg_photos = df["PhotoAmt"].mean()

kpis = {
    "Total mascotas"           : f"{total:,}",
    "Perros"                   : f"{n_dogs:,} ({n_dogs/total*100:.1f}%)",
    "Gatos"                    : f"{n_cats:,} ({n_cats/total*100:.1f}%)",
    "AdoptionSpeed promedio"   : f"{avg_speed:.2f} / 4",
    "Adoptados en ≤7 días"     : f"{pct_fast:.1f}%",
    "No adoptados (+100 días)" : f"{pct_slow:.1f}%",
    "Fotos promedio"           : f"{avg_photos:.1f}",
    "Estados representados"    : n_states,
}
pd.DataFrame(kpis.items(), columns=["Indicador","Valor"])


,Indicador,Valor
0,Total mascotas,"14,993"
1,Perros,"8,132 (54.2%)"
2,Gatos,"6,861 (45.8%)"
3,AdoptionSpeed promedio,2.52 / 4
4,Adoptados en ≤7 días,23.3%
5,No adoptados (+100 días),28.0%
6,Fotos promedio,3.9
7,Estados representados,14


## 🔍 4 · Calidad de Datos y Valores Nulos

Solo `Name` (1.265 registros) y `Description` (13) tienen nulos. El dataset tiene **calidad excelente**.

In [19]:
null_df = (df.isnull().sum()
             .reset_index()
             .rename(columns={"index":"Columna",0:"Nulos"}))
null_df.columns = ["Columna","Nulos"]
null_df["% Nulos"] = (null_df["Nulos"] / len(df) * 100).round(2)
null_df = null_df[null_df["Nulos"] > 0].sort_values("Nulos", ascending=True)
display(null_df.reset_index(drop=True))

fig = go.Figure(go.Bar(
    x=null_df["% Nulos"], y=null_df["Columna"], orientation="h",
    marker=dict(color=null_df["% Nulos"], colorscale="Reds", showscale=False,
                line=dict(color="#ccc", width=0.5)),
    text=[f"{n} registros ({p:.1f}%)" for n,p in zip(null_df["Nulos"],null_df["% Nulos"])],
    textposition="outside",
))
fig.update_layout(title="Valores Nulos por Columna", height=250,
                  plot_bgcolor=BG_COLOR, paper_bgcolor="white",
                  xaxis_title="% Nulos", yaxis_title="")
fig.show()


,Columna,Nulos,% Nulos
0,BreedName1,5,0.03
1,Description,13,0.09
2,Name,1265,8.44


## 📊 5 · Análisis de Frecuencias con Histogramas

Todos los histogramas muestran la distribución **coloreada por `AdoptionSpeed`** para detectar patrones con la variable objetivo.

### 5.1 · Edad de las mascotas

In [20]:
fig = go.Figure()
for spd in sorted(df["AdoptionSpeed"].unique()):
    sub = df[df["AdoptionSpeed"]==spd]["Age"]
    fig.add_trace(go.Histogram(
        x=sub, nbinsx=50, name=SPEED_LABELS[spd],
        marker_color=SPEED_COLORS[spd], opacity=0.65,
    ))
fig.update_layout(
    barmode="overlay",
    title="Distribución de Edad por AdoptionSpeed",
    xaxis_title="Edad (meses)", yaxis_title="Frecuencia",
    height=450, plot_bgcolor=BG_COLOR, paper_bgcolor="white",
    legend_title="AdoptionSpeed"
)
fig.show()


### 5.2 · Fee de adopción

In [21]:
df_fee = df[df["Fee"] <= 500]
fig = go.Figure()
for spd in sorted(df["AdoptionSpeed"].unique()):
    sub = df_fee[df_fee["AdoptionSpeed"]==spd]["Fee"]
    fig.add_trace(go.Histogram(
        x=sub, nbinsx=40, name=SPEED_LABELS[spd],
        marker_color=SPEED_COLORS[spd], opacity=0.65,
    ))
fig.update_layout(
    barmode="overlay",
    title="Distribución del Fee por AdoptionSpeed (Fee ≤ 500 MYR)",
    xaxis_title="Fee (MYR)", yaxis_title="Frecuencia",
    height=450, plot_bgcolor=BG_COLOR, paper_bgcolor="white",
    legend_title="AdoptionSpeed"
)
fig.show()


### 5.3 · Cantidad de fotos

In [22]:
fig = go.Figure()
for spd in sorted(df["AdoptionSpeed"].unique()):
    sub = df[df["AdoptionSpeed"]==spd]["PhotoAmt"]
    fig.add_trace(go.Histogram(
        x=sub, nbinsx=30, name=SPEED_LABELS[spd],
        marker_color=SPEED_COLORS[spd], opacity=0.65,
    ))
fig.update_layout(
    barmode="overlay",
    title="Distribución de Fotos por AdoptionSpeed",
    xaxis_title="Cantidad de fotos", yaxis_title="Frecuencia",
    height=450, plot_bgcolor=BG_COLOR, paper_bgcolor="white",
    legend_title="AdoptionSpeed"
)
fig.show()


### 5.4 · Frecuencias de variables categóricas clave

In [23]:
# Subplots: Tipo | Género | Salud | Pelaje
fig = make_subplots(rows=2, cols=2,
    subplot_titles=("Tipo de Mascota","Género","Estado de Salud","Longitud de Pelaje"))

for r, c, col_name, labels in [
    (1,1,"Type_Label",  None),
    (1,2,"Gender_Label",None),
    (2,1,"Health_Label",None),
    (2,2,"Fur_Label",   None),
]:
    vc = df[col_name].value_counts().reset_index()
    vc.columns = ["cat","count"]
    fig.add_trace(go.Bar(
        x=vc["cat"], y=vc["count"],
        marker_color=PALETTE[:len(vc)],
        text=vc["count"], textposition="outside",
        showlegend=False,
    ), row=r, col=c)

fig.update_layout(height=600, paper_bgcolor="white", plot_bgcolor=BG_COLOR,
                  title_text="Frecuencias de Variables Categóricas")
fig.show()


### 5.5 · Histograma de AdoptionSpeed

In [24]:
speed_counts = df["AdoptionSpeed"].value_counts().sort_index().reset_index()
speed_counts.columns = ["Speed","Count"]
speed_counts["Label"]   = speed_counts["Speed"].map(SPEED_LABELS)
speed_counts["Color"]   = speed_counts["Speed"].map(SPEED_COLORS)
speed_counts["Pct"]     = (speed_counts["Count"] / len(df) * 100).round(1)

fig = go.Figure(go.Bar(
    x=speed_counts["Label"],
    y=speed_counts["Count"],
    marker=dict(color=speed_counts["Color"],
                line=dict(color="white", width=1.5)),
    text=[f"{c:,}<br>({p}%)" for c,p in zip(speed_counts["Count"],speed_counts["Pct"])],
    textposition="outside",
))
fig.update_layout(
    title="Distribución de AdoptionSpeed (variable objetivo)",
    xaxis_title="Velocidad de Adopción", yaxis_title="Cantidad de Mascotas",
    height=480, plot_bgcolor=BG_COLOR, paper_bgcolor="white", showlegend=False,
)
fig.show()


### 5.6 · AdoptionSpeed por Tipo de Mascota

In [25]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Perros 🐕","Gatos 🐈"))

for col_i, (tipo, tipo_num) in enumerate([("Perro 🐕",1),("Gato 🐈",2)], start=1):
    sub = df[df["Type"]==tipo_num]
    vc  = sub["AdoptionSpeed"].value_counts().sort_index().reset_index()
    vc.columns = ["Speed","Count"]
    vc["Label"] = vc["Speed"].map(SPEED_LABELS)
    vc["Color"] = vc["Speed"].map(SPEED_COLORS)
    fig.add_trace(go.Bar(
        x=vc["Label"], y=vc["Count"],
        marker_color=vc["Color"],
        text=vc["Count"], textposition="outside",
        showlegend=False,
    ), row=1, col=col_i)

fig.update_layout(height=460, paper_bgcolor="white", plot_bgcolor=BG_COLOR,
                  title_text="AdoptionSpeed por Tipo de Mascota")
fig.update_xaxes(tickangle=-20)
fig.show()


## 🎯 6 · Análisis Profundo de AdoptionSpeed

### 6.1 · AdoptionSpeed promedio por Raza (Top 15 más rápidas y lentas)

In [26]:
breed_speed = (df.groupby("BreedName1")["AdoptionSpeed"]
                 .agg(Promedio="mean", Count="count")
                 .reset_index()
                 .query("Count >= 30")
                 .sort_values("Promedio"))

top_fast = breed_speed.head(10).copy()
top_slow = breed_speed.tail(10).copy()
combined = pd.concat([top_fast, top_slow]).reset_index(drop=True)
combined["Color"] = combined["Promedio"].apply(
    lambda x: "#2ecc71" if x < 2.3 else ("#e74c3c" if x > 2.8 else "#f39c12"))

fig = go.Figure(go.Bar(
    x=combined["Promedio"], y=combined["BreedName1"],
    orientation="h",
    marker=dict(color=combined["Color"], line=dict(color="white",width=0.8)),
    text=[f"{v:.2f} (n={n})" for v,n in zip(combined["Promedio"],combined["Count"])],
    textposition="outside",
))
fig.add_vline(x=df["AdoptionSpeed"].mean(), line_dash="dash",
              line_color="gray", annotation_text="Promedio global",
              annotation_position="top")
fig.update_layout(
    title="AdoptionSpeed Promedio por Raza (razas con ≥30 registros)",
    xaxis_title="AdoptionSpeed Promedio (menor = más rápido)",
    height=600, plot_bgcolor=BG_COLOR, paper_bgcolor="white",
    yaxis=dict(autorange="reversed")
)
fig.show()


### 6.2 · AdoptionSpeed por Color, Salud y Tamaño

In [27]:
fig = make_subplots(rows=1, cols=3,
    subplot_titles=("Por Color Primario","Por Estado de Salud","Por Tamaño de Madurez"))

for col_i, col_name in enumerate(["ColorName1","Health_Label","Maturity_Label"], start=1):
    grp = (df.groupby(col_name)["AdoptionSpeed"]
             .mean().sort_values().reset_index())
    grp.columns = ["cat","mean_speed"]
    grp["color"] = grp["mean_speed"].apply(
        lambda x: "#2ecc71" if x < 2.4 else ("#e74c3c" if x > 2.7 else "#f39c12"))
    fig.add_trace(go.Bar(
        x=grp["cat"], y=grp["mean_speed"],
        marker_color=grp["color"],
        text=[f"{v:.2f}" for v in grp["mean_speed"]],
        textposition="outside", showlegend=False,
    ), row=1, col=col_i)

fig.update_layout(height=440, paper_bgcolor="white", plot_bgcolor=BG_COLOR,
                  title_text="AdoptionSpeed por Color, Salud y Tamaño")
fig.update_xaxes(tickangle=-30)
fig.show()


### 6.3 · Impacto de Fotos y Videos en AdoptionSpeed

In [28]:
photo_speed = (df.groupby("AdoptionSpeed")["PhotoAmt"]
                 .mean().reset_index())
photo_speed["Label"] = photo_speed["AdoptionSpeed"].map(SPEED_LABELS)
photo_speed["Color"] = photo_speed["AdoptionSpeed"].map(SPEED_COLORS)

fig = go.Figure(go.Bar(
    x=photo_speed["Label"], y=photo_speed["PhotoAmt"],
    marker_color=photo_speed["Color"],
    text=[f"{v:.2f}" for v in photo_speed["PhotoAmt"]],
    textposition="outside",
))
fig.update_layout(
    title="Promedio de Fotos por Nivel de AdoptionSpeed",
    xaxis_title="AdoptionSpeed", yaxis_title="Fotos promedio",
    height=440, plot_bgcolor=BG_COLOR, paper_bgcolor="white",
)
fig.show()


## 🎻 7 · Gráficos de Violín — Distribuciones por Variable Categórica

> Los violins muestran la **forma completa de la distribución** de `AdoptionSpeed` dentro de cada categoría, revelando asimetrías y multimodalidad que los box plots ocultan.

### 7.1 · Tipo de mascota

In [29]:
fig = go.Figure()
for tipo, color in [("Perro 🐕", COLOR_DOG), ("Gato 🐈", COLOR_CAT)]:
    sub = df[df["Type_Label"]==tipo]["AdoptionSpeed"]
    fig.add_trace(go.Violin(
        y=sub, name=tipo,
        box_visible=True, meanline_visible=True,
        fillcolor=color, opacity=0.7,
        line_color="white", points="outliers",
    ))
fig.update_layout(
    title="Distribución de AdoptionSpeed por Tipo de Mascota",
    yaxis_title="AdoptionSpeed", height=500,
    plot_bgcolor=BG_COLOR, paper_bgcolor="white",
    violingap=0.3, violinmode="overlay",
)
fig.show()


### 7.2 · Estado de salud

In [30]:
health_colors = {"Saludable":"#2ecc71","Lesión leve":"#f39c12","Lesión grave":"#e74c3c"}
fig = go.Figure()
for salud, color in health_colors.items():
    sub = df[df["Health_Label"]==salud]["AdoptionSpeed"]
    if len(sub) > 0:
        fig.add_trace(go.Violin(
            y=sub, name=f"{salud} (n={len(sub):,})",
            box_visible=True, meanline_visible=True,
            fillcolor=color, opacity=0.75,
            line_color="white", points="outliers",
        ))
fig.update_layout(
    title="Distribución de AdoptionSpeed por Estado de Salud",
    yaxis_title="AdoptionSpeed", height=500,
    plot_bgcolor=BG_COLOR, paper_bgcolor="white",
)
fig.show()


### 7.3 · Vacunación

In [31]:
vax_colors = {"Sí":"#27ae60","No":"#e74c3c","No seguro":"#95a5a6"}
fig = go.Figure()
for vax, color in vax_colors.items():
    sub = df[df["Vaccinated_Label"]==vax]["AdoptionSpeed"]
    fig.add_trace(go.Violin(
        y=sub, name=f"{vax} (n={len(sub):,})",
        box_visible=True, meanline_visible=True,
        fillcolor=color, opacity=0.75,
        line_color="white", points="outliers",
    ))
fig.update_layout(
    title="Distribución de AdoptionSpeed por Vacunación",
    yaxis_title="AdoptionSpeed", height=500,
    plot_bgcolor=BG_COLOR, paper_bgcolor="white",
)
fig.show()


### 7.4 · Longitud de pelaje

In [32]:
fur_colors = {"Corto":"#3498db","Mediano":"#9b59b6","Largo":"#e67e22"}
fig = go.Figure()
for fur, color in fur_colors.items():
    sub = df[df["Fur_Label"]==fur]["AdoptionSpeed"]
    fig.add_trace(go.Violin(
        y=sub, name=f"{fur} (n={len(sub):,})",
        box_visible=True, meanline_visible=True,
        fillcolor=color, opacity=0.75,
        line_color="white", points="outliers",
    ))
fig.update_layout(
    title="Distribución de AdoptionSpeed por Longitud de Pelaje",
    yaxis_title="AdoptionSpeed", height=500,
    plot_bgcolor=BG_COLOR, paper_bgcolor="white",
)
fig.show()


### 7.5 · Esterilización

In [33]:
steril_colors = {"Sí":"#1abc9c","No":"#e74c3c","No seguro":"#95a5a6"}
fig = go.Figure()
for est, color in steril_colors.items():
    sub = df[df["Sterilized_Label"]==est]["AdoptionSpeed"]
    fig.add_trace(go.Violin(
        y=sub, name=f"{est} (n={len(sub):,})",
        box_visible=True, meanline_visible=True,
        fillcolor=color, opacity=0.75,
        line_color="white", points="outliers",
    ))
fig.update_layout(
    title="Distribución de AdoptionSpeed por Esterilización",
    yaxis_title="AdoptionSpeed", height=500,
    plot_bgcolor=BG_COLOR, paper_bgcolor="white",
)
fig.show()


### 7.6 · Categoría de Fee

In [ ]:
fee_order  = ["Gratis","Bajo (1-50)","Medio (51-200)","Alto (201-500)","Premium (>500)"]
fee_colors = ["#2ecc71","#3498db","#f39c12","#e67e22","#e74c3c"]

fig = go.Figure()
for cat, color in zip(fee_order, fee_colors):
    sub = df[df["FeeCategory"]==cat]["AdoptionSpeed"]
    if len(sub) > 0:
        fig.add_trace(go.Violin(
            y=sub, name=f"{cat} (n={len(sub):,})",
            box_visible=True, meanline_visible=True,
            fillcolor=color, opacity=0.75,
            line_color="white", points="outliers",
        ))
fig.update_layout(
    title="Distribución de AdoptionSpeed por Categoría de Fee",
    yaxis_title="AdoptionSpeed", height=520,
    plot_bgcolor=BG_COLOR, paper_bgcolor="white",
)
fig.show()


### 7.7 · Tamaño de madurez

In [ ]:
fig = go.Figure()
for i, (mat, color) in enumerate(zip(
    ["Pequeño","Mediano","Grande","Extra grande"],
    ["#3498db","#2ecc71","#f39c12","#e74c3c"]
)):
    sub = df[df["Maturity_Label"]==mat]["AdoptionSpeed"]
    if len(sub) > 0:
        fig.add_trace(go.Violin(
            y=sub, name=f"{mat} (n={len(sub):,})",
            box_visible=True, meanline_visible=True,
            fillcolor=color, opacity=0.75,
            line_color="white", points="outliers",
        ))
fig.update_layout(
    title="Distribución de AdoptionSpeed por Tamaño de Madurez",
    yaxis_title="AdoptionSpeed", height=500,
    plot_bgcolor=BG_COLOR, paper_bgcolor="white",
)
fig.show()


## 🗺️ 8 · Heatmaps de Correlación — Simple y Parcial

> El **heatmap simple** muestra la correlación Pearson entre pares.  
> El **heatmap parcial** elimina el efecto de todas las demás variables, revelando relaciones directas puras.

In [34]:
CORR_COLS = ["Age","Fee","Quantity","PhotoAmt","VideoAmt",
             "Vaccinated","Dewormed","Sterilized","Health",
             "MaturitySize","FurLength","AdoptionSpeed"]

data_c   = df[CORR_COLS].dropna()
corr_mat = data_c.corr().round(3)

# Correlación parcial via precision matrix
prec  = np.linalg.inv(corr_mat.values)
n_var = len(CORR_COLS)
pcorr = np.zeros((n_var, n_var))
for i in range(n_var):
    for j in range(n_var):
        if i == j:
            pcorr[i,j] = 1.0
        else:
            denom = np.sqrt(prec[i,i] * prec[j,j])
            pcorr[i,j] = -prec[i,j] / denom if denom != 0 else 0.0

pcorr_df = pd.DataFrame(pcorr, index=CORR_COLS, columns=CORR_COLS).round(3)

print("✅ Matrices calculadas")
print(f"Correlación con AdoptionSpeed (simple):")
corr_target = corr_mat["AdoptionSpeed"].drop("AdoptionSpeed").sort_values()
for var, val in corr_target.items():
    bar = "█" * int(abs(val)*40)
    sign = "+" if val > 0 else "-"
    print(f"  {var:15} {sign}{abs(val):.3f}  {bar}")


✅ Matrices calculadas
Correlación con AdoptionSpeed (simple):
  FurLength       -0.091  ███
  Sterilized      -0.083  ███
  Vaccinated      -0.059  ██
  PhotoAmt        -0.023  
  Dewormed        -0.013  
  Fee             -0.004  
  VideoAmt        -0.001  
  Health          +0.029  █
  MaturitySize    +0.046  █
  Quantity        +0.063  ██
  Age             +0.101  ████


In [35]:
# Heatmap Correlación Simple
fig = go.Figure(go.Heatmap(
    z=corr_mat.values,
    x=CORR_COLS, y=CORR_COLS,
    colorscale="RdBu_r", zmid=0, zmin=-1, zmax=1,
    text=corr_mat.round(2).values,
    texttemplate="%{text}", textfont=dict(size=9),
    colorbar=dict(title="r", len=0.8),
    hoverongaps=False,
))
fig.update_layout(
    title="Heatmap de Correlación Simple (Pearson)",
    height=580, paper_bgcolor="white",
    xaxis=dict(tickangle=-40, tickfont=dict(size=10)),
    yaxis=dict(tickfont=dict(size=10)),
    margin=dict(l=10, r=10, t=60, b=10),
)
fig.show()


In [38]:
# Heatmap Correlación Parcial
fig = go.Figure(go.Heatmap(
    z=pcorr_df.values,
    x=CORR_COLS, y=CORR_COLS,
    colorscale="RdBu_r", zmid=0, zmin=-1, zmax=1,
    text=pcorr_df.round(2).values,
    texttemplate="%{text}", textfont=dict(size=9),
    colorbar=dict(title="rp", len=0.8),
    hoverongaps=False,
))
fig.update_layout(
    title="Heatmap de Correlación Parcial (controlando todas las demás variables)",
    height=580, paper_bgcolor="white",
    xaxis=dict(tickangle=-40, tickfont=dict(size=10)),
    yaxis=dict(tickfont=dict(size=10)),
    margin=dict(l=10, r=10, t=60, b=10),
)
fig.show()


In [39]:
# Comparación Simple vs Parcial para AdoptionSpeed
pairs = []
target = "AdoptionSpeed"
for var in CORR_COLS:
    if var == target: continue
    r_s = round(corr_mat.loc[var, target], 3)
    r_p = round(pcorr_df.loc[var, target], 3)
    pairs.append({"Variable":var, "r simple":r_s, "r parcial":r_p,
                  "Diferencia":round(abs(r_s)-abs(r_p),3)})

pairs_df = pd.DataFrame(pairs).sort_values("r simple")
display(pairs_df.reset_index(drop=True))

fig = go.Figure()
fig.add_trace(go.Bar(
    name="r simple", x=pairs_df["Variable"], y=pairs_df["r simple"],
    marker_color=[("#4E79A7" if v>=0 else "#E15759") for v in pairs_df["r simple"]],
    text=[f"{v:+.3f}" for v in pairs_df["r simple"]], textposition="outside",
))
fig.add_trace(go.Bar(
    name="r parcial", x=pairs_df["Variable"], y=pairs_df["r parcial"],
    marker_color=[("#59A14F" if v>=0 else "#F28E2B") for v in pairs_df["r parcial"]],
    text=[f"{v:+.3f}" for v in pairs_df["r parcial"]], textposition="outside",
))
fig.add_hline(y=0, line_color="black", line_width=1)
fig.update_layout(
    barmode="group",
    title="Correlación Simple vs Parcial con AdoptionSpeed",
    xaxis_tickangle=-35,
    yaxis=dict(title="Correlación", range=[-0.5, 0.5]),
    height=500, plot_bgcolor=BG_COLOR, paper_bgcolor="white",
    legend=dict(x=0.75, y=0.98),
)
fig.show()


,Variable,r simple,r parcial,Diferencia
0,FurLength,-0.09,-0.11,-0.02
1,Sterilized,-0.08,-0.05,0.03
2,Vaccinated,-0.06,-0.04,0.01
3,PhotoAmt,-0.02,-0.03,-0.01
4,Dewormed,-0.01,0.04,-0.03
5,Fee,-0.00,0.00,0.00
6,VideoAmt,-0.00,0.00,-0.00
7,Health,0.03,0.03,-0.00
8,MaturitySize,0.05,0.04,0.00
9,Quantity,0.06,0.08,-0.02


## 🔵 9 · Pairplot Interactivo — 8 Variables vs AdoptionSpeed

> Cada celda muestra la relación entre dos variables, **coloreada por AdoptionSpeed**.  
> La diagonal muestra histogramas de cada variable.  
> Permite detectar de un vistazo qué combinaciones de variables separan mejor los niveles de velocidad.

In [40]:
PAIR_COLS = ["Age","Fee","PhotoAmt","VideoAmt",
             "Quantity","Health","MaturitySize","FurLength"]

# Sample para performance (pairplot con 14k puntos es pesado)
df_sample = df.sample(n=min(3000, len(df)), random_state=42).copy()
df_sample["Speed_Label"] = df_sample["AdoptionSpeed"].map(SPEED_LABELS)

fig = px.scatter_matrix(
    df_sample,
    dimensions=PAIR_COLS,
    color="Speed_Label",
    color_discrete_map=SPEED_LABELS | {v:k for k,v in
        {SPEED_LABELS[k]:SPEED_COLORS[k] for k in SPEED_COLORS}.items()},
    color_discrete_sequence=[SPEED_COLORS[i] for i in sorted(SPEED_COLORS)],
    opacity=0.4,
    title="Pairplot: 8 Variables Coloreadas por AdoptionSpeed",
    labels={c: c.replace("MaturitySize","MatSize") for c in PAIR_COLS},
)
fig.update_traces(
    marker=dict(size=3),
    diagonal_visible=True,
    showupperhalf=False,
)
fig.update_layout(
    height=900, paper_bgcolor="white",
    legend_title="AdoptionSpeed",
    font=dict(size=10),
)
fig.show()


### 9.1 · Scatter detallado: Age vs AdoptionSpeed

In [41]:
# Jitter para ver mejor la distribución discreta
np.random.seed(42)
df_j = df.sample(n=3000, random_state=42).copy()
df_j["Speed_jitter"] = df_j["AdoptionSpeed"] + np.random.uniform(-0.25, 0.25, len(df_j))

fig = go.Figure()
for spd in sorted(df["AdoptionSpeed"].unique()):
    sub = df_j[df_j["AdoptionSpeed"]==spd]
    fig.add_trace(go.Scatter(
        x=sub["Age"], y=sub["Speed_jitter"],
        mode="markers", name=SPEED_LABELS[spd],
        marker=dict(color=SPEED_COLORS[spd], opacity=0.45, size=5),
    ))
fig.update_layout(
    title="Age vs AdoptionSpeed (con jitter para legibilidad)",
    xaxis_title="Edad (meses)", yaxis_title="AdoptionSpeed",
    height=480, plot_bgcolor=BG_COLOR, paper_bgcolor="white",
)
fig.show()


## 🗺️ 10 · Análisis Geográfico — AdoptionSpeed por Estado

In [42]:
STATE_COORDS = {
    "Kuala Lumpur":(3.1390,101.6869), "Terengganu":(5.3117,103.1324),
    "Labuan":(5.2831,115.2308),       "Melaka":(2.1896,102.2501),
    "Perak":(4.5921,101.0901),        "Negeri Sembilan":(2.7258,102.2378),
    "Kedah":(6.1184,100.3685),        "Kelantan":(6.1254,102.2381),
    "Pahang":(3.8126,103.3256),       "Sarawak":(1.5533,110.3592),
    "Pulau Pinang":(5.4141,100.3288), "Perlis":(6.4449,100.2048),
    "Johor":(1.9344,103.3587),        "Sabah":(5.9788,116.0753),
    "Selangor":(3.0738,101.5183),
}

state_stats = (df.groupby("StateName").agg(
    Total          =("PetID",         "count"),
    Speed_Promedio =("AdoptionSpeed",  "mean"),
    Speed_Mediana  =("AdoptionSpeed",  "median"),
    Pct_Rapidos    =("AdoptionSpeed",  lambda x: (x<=1).mean()*100),
    Pct_Lentos     =("AdoptionSpeed",  lambda x: (x==4).mean()*100),
    Perros         =("Type",          lambda x: (x==1).sum()),
    Gatos          =("Type",          lambda x: (x==2).sum()),
).reset_index())

state_stats["Speed_Promedio"] = state_stats["Speed_Promedio"].round(3)
state_stats["Pct_Rapidos"]    = state_stats["Pct_Rapidos"].round(1)
state_stats["Pct_Lentos"]     = state_stats["Pct_Lentos"].round(1)
state_stats["Lat"] = state_stats["StateName"].map(lambda s: STATE_COORDS.get(s,(None,None))[0])
state_stats["Lon"] = state_stats["StateName"].map(lambda s: STATE_COORDS.get(s,(None,None))[1])
state_stats["BubbleSize"] = state_stats["Total"] / state_stats["Total"].max() * 55 + 8

display(state_stats[["StateName","Total","Speed_Promedio","Pct_Rapidos","Pct_Lentos"]].sort_values("Speed_Promedio"))


,StateName,Total,Speed_Promedio,Pct_Rapidos,Pct_Lentos
7,Pahang,85,2.32,37.60,27.10
2,Kelantan,15,2.40,33.30,40.00
0,Johor,507,2.41,26.80,26.00
12,Selangor,8714,2.45,24.40,24.70
3,Kuala Lumpur,3845,2.54,24.10,31.00
10,Sabah,22,2.54,31.80,36.40
13,Terengganu,26,2.58,34.60,34.60
1,Kedah,110,2.68,15.50,32.70
9,Pulau Pinang,843,2.78,15.40,35.60
8,Perak,420,2.82,12.10,33.60


In [43]:
# Mapa: AdoptionSpeed promedio por estado
fig = px.scatter_geo(
    state_stats, lat="Lat", lon="Lon",
    size="BubbleSize", size_max=55,
    color="Speed_Promedio",
    color_continuous_scale="RdYlGn_r",
    hover_name="StateName",
    hover_data={"Speed_Promedio":True,"Pct_Rapidos":True,
                "Pct_Lentos":True,"Total":True,
                "Lat":False,"Lon":False,"BubbleSize":False},
    text="StateName",
    title="🗺️ AdoptionSpeed Promedio por Estado (verde = más rápido)",
)
fig.update_traces(textposition="top center",
    marker=dict(opacity=0.88, line=dict(color="white",width=1.5)))
fig.update_geos(
    resolution=50, showcountries=True, countrycolor="#AAAAAA",
    showcoastlines=True, showland=True, landcolor="#EEF0E5",
    showocean=True, oceancolor="#C8DFF5",
    lataxis_range=[-2,9], lonaxis_range=[98,120],
    projection_type="mercator",
)
fig.update_layout(height=560, paper_bgcolor="white",
                  coloraxis_colorbar=dict(title="Speed Prom."),
                  margin=dict(l=0,r=0,t=50,b=10))
fig.show()


In [44]:
# Mapa: % adoptados rápido (≤7 días)
fig = px.scatter_geo(
    state_stats, lat="Lat", lon="Lon",
    size="BubbleSize", size_max=55,
    color="Pct_Rapidos",
    color_continuous_scale="Greens",
    hover_name="StateName",
    hover_data={"Pct_Rapidos":True,"Speed_Promedio":True,
                "Total":True,"Lat":False,"Lon":False,"BubbleSize":False},
    text="StateName",
    title="🗺️ % Mascotas Adoptadas en ≤7 días por Estado",
)
fig.update_traces(textposition="top center",
    marker=dict(opacity=0.88, line=dict(color="white",width=1.5)))
fig.update_geos(
    resolution=50, showcountries=True, countrycolor="#AAAAAA",
    showcoastlines=True, showland=True, landcolor="#EEF0E5",
    showocean=True, oceancolor="#C8DFF5",
    lataxis_range=[-2,9], lonaxis_range=[98,120],
    projection_type="mercator",
)
fig.update_layout(height=560, paper_bgcolor="white",
                  coloraxis_colorbar=dict(title="% Rápidos"),
                  margin=dict(l=0,r=0,t=50,b=10))
fig.show()


In [45]:
# Bar chart comparativo por estado
fig = go.Figure()
ss = state_stats.sort_values("Speed_Promedio")
fig.add_trace(go.Bar(
    name="% Adoptados ≤7 días",
    x=ss["StateName"], y=ss["Pct_Rapidos"],
    marker_color="#2ecc71", text=[f"{v:.1f}%" for v in ss["Pct_Rapidos"]],
    textposition="outside",
))
fig.add_trace(go.Bar(
    name="% No adoptados (+100 días)",
    x=ss["StateName"], y=ss["Pct_Lentos"],
    marker_color="#e74c3c", text=[f"{v:.1f}%" for v in ss["Pct_Lentos"]],
    textposition="outside",
))
fig.update_layout(
    barmode="group",
    title="Velocidad de Adopción por Estado — Rápidos vs Lentos",
    xaxis_tickangle=-35, yaxis_title="%",
    height=500, plot_bgcolor=BG_COLOR, paper_bgcolor="white",
)
fig.show()


## 📌 11 · Conclusiones Principales

| # | Hallazgo | Detalle |
|---|---|---|
| 1 | ⏱️ **28% no se adoptan en 100 días** | AdoptionSpeed=4 es la categoría más frecuente |
| 2 | 🐈 **Los gatos se adoptan más rápido** | Speed promedio: gatos 2.38 vs perros 2.62 |
| 3 | 👶 **La edad es el predictor más fuerte** | r=+0.10 con AdoptionSpeed — más viejo = más lento |
| 4 | 🎻 **Mascotas esterilizadas y vacunadas** tienen distribuciones más cargadas hacia velocidades bajas | |
| 5 | 📸 **Las fotos tienen un efecto no lineal** | Speed=2 tiene el máximo de fotos; Speed=4 tiene pocas |
| 6 | 🏆 **Razas más rápidas:** Persas, Poodles, Shih Tzu, Golden Retriever | Speed prom. ~2.0 |
| 7 | 🗺️ **Pahang y Kelantan** son los estados con adopción más rápida | Speed prom. ~2.3 |
| 8 | 🗺️ **Sarawak y Melaka** son los más lentos | Speed prom. ~3.1–3.3 |
| 9 | 🔬 **Correlaciones parciales bajas** | Ninguna variable tiene rp > 0.15 con AdoptionSpeed — el fenómeno es multivariado |
| 10 | 🎻 **El pelaje corto se adopta más rápido** | Violin de FurLength muestra masa concentrada en Speed 1-2 |
